In [1]:
from pathlib import Path

In [4]:
file_path = Path("../data/raw/data.zip")

size_byte = file_path.stat().st_size

print(f"{size_byte / (1024**2):.2f} MB")
print(f"{size_byte / (1024**3):.2f} GB")

4.99 MB
0.00 GB


In [ ]:
import zipfile

In [7]:
with zipfile.ZipFile(file_path, "r") as zip_ref:
    zip_ref.extractall(file_path.parent)

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/historical_data.csv")
df["created_at"] = pd.to_datetime(df["created_at"])
df["actual_delivery_time"] = pd.to_datetime(df["actual_delivery_time"])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 16 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   market_id                                     196441 non-null  float64       
 1   created_at                                    197428 non-null  datetime64[us]
 2   actual_delivery_time                          197421 non-null  datetime64[us]
 3   store_id                                      197428 non-null  int64         
 4   store_primary_category                        192668 non-null  str           
 5   order_protocol                                196433 non-null  float64       
 6   total_items                                   197428 non-null  int64         
 7   subtotal                                      197428 non-null  int64         
 8   num_distinct_items                            197428 non-null  in

In [4]:
df['created_at'].dt.to_period("M").value_counts().sort_index()

created_at
2014-10         1
2015-01     68482
2015-02    128945
Freq: M, Name: count, dtype: int64

In [ ]:
df['created_at'].dt.hour

0          6
1         10
2         22
3          3
4         15
          ..
197423    17
197424    13
197425    24
197426     1
197427     8
Name: created_at, Length: 197428, dtype: int32

In [9]:
df.head()

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


## **Missing Features**

In [10]:
df.isna().sum()

market_id                                         987
created_at                                          0
actual_delivery_time                                7
store_id                                            0
store_primary_category                           4760
order_protocol                                    995
total_items                                         0
subtotal                                            0
num_distinct_items                                  0
min_item_price                                      0
max_item_price                                      0
total_onshift_dashers                           16262
total_busy_dashers                              16262
total_outstanding_orders                        16262
estimated_order_place_duration                      0
estimated_store_to_consumer_driving_duration      526
dtype: int64

In [11]:
df.isna().mean()


market_id                                       0.004999
created_at                                      0.000000
actual_delivery_time                            0.000035
store_id                                        0.000000
store_primary_category                          0.024110
order_protocol                                  0.005040
total_items                                     0.000000
subtotal                                        0.000000
num_distinct_items                              0.000000
min_item_price                                  0.000000
max_item_price                                  0.000000
total_onshift_dashers                           0.082369
total_busy_dashers                              0.082369
total_outstanding_orders                        0.082369
estimated_order_place_duration                  0.000000
estimated_store_to_consumer_driving_duration    0.002664
dtype: float64

- All money value are in cents
- All time are in seconds
- Each row in this file corresponds to one unique delivery.

Columns definitions:

Time features
market_id: A city/region in which DoorDash operates, e.g., Los Angeles, given in the data as an id

created_at: Timestamp in UTC when the order was submitted by the consumer to DoorDash. (Note this timestamp is in UTC, but in case you need it, the actual timezone of the region was US/Pacific)

actual_delivery_time: Timestamp in UTC when the order was delivered to the consumer

Store features
store_id: an id representing the restaurant the order was submitted for

store_primary_category: cuisine category of the restaurant, e.g., italian, asian

order_protocol: a store can receive orders from DoorDash through many modes. This field represents an id denoting the protocol

Order features
total_items: total number of items in the order

subtotal: total value of the order submitted (in cents)

num_distinct_items: number of distinct items included in the order

min_item_price: price of the item with the least cost in the order (in cents)

max_item_price: price of the item with the highest cost in the order (in cents)

Market features
DoorDash being a marketplace, we have information on the state of marketplace when the order is placed, that can be used to estimate delivery time. The following features are values at the time of created_at (order submission time):

total_onshift_dashers: Number of available dashers who are within 10 miles of the store at the time of order creation

total_busy_dashers: Subset of above total_onshift_dashers who are currently working on an order

total_outstanding_orders: Number of orders within 10 miles of this order that are currently being processed.

Predictions from other models
We have predictions from other models for various stages of delivery process that we can use:
estimated_order_place_duration: Estimated time for the restaurant to receive the order from DoorDash (in seconds)

estimated_store_to_consumer_driving_duration: Estimated travel time between store and consumer (in seconds)

In [ ]:
# Investigate market columns
market_cols = [
    "total_onshift_dashers",
    "total_busy_dashers",
    "total_outstanding_orders",
]

df[market_cols].isna().all(axis=1).sum()
# yes same missing rows

np.int64(16262)

In [ ]:
df.groupby("store_id")["estimated_store_to_consumer_driving_duration"].apply(
    lambda x: x.isna().sum()
).gt(0).value_counts()


estimated_store_to_consumer_driving_duration
False    6293
True      450
Name: count, dtype: int64

Understanding why missing then possible fix:
- `Market features`. confirmed all has same rows missing.
check if same rows. if missing in same rows fill with 0. i think it means no total_onshift_dashers, total_busy_dashers, and total_outstanding_orders repectively.
But If the upstream marketplace system failed to provide the snapshot, filling all three with zero would manufacture an operational state that might never have happened.
fix:
inpute

- `estimated_store_to_consumer_driving_duration` a prediction from another model. i don't think it's safe to blindly inpute.
a few useful things:
- 526 missing observations out of the dataset.
- 450 stores have at least one missing observation.
- More than 6,000 stores are involved in the dataset.
- Missingness is relatively sparse overall.
fix:
1. Create a missingness indicator.
2. Inpute fallback hierarchy: store-level median -> maket-level mdeian -> global median
3. Calculate all medians using training data only.
enough obersavation = 5 (not saying it's optimal thorugh experimentation i would get optimal.)

- how can we not have a `market_id`? could be operational issue
fix:
add new unknown category

- `actual_delivery_time` target unavailable. I need groundttruth to evaluate model performance.
fix:
drop those 7 rows.

- `order_protocol` doc said a store *can*, so it can not.
Could be it received no protocol, or the protocol wasn't recorded or is unknown
fix:
add new unknown category

- `store_primary_category`
A restaurant not having a cuisine classification doesn't mean it has no cuisine.
So I'd interpret missingness as an unknown or unavailable category, not necessarily a restaurant without a cuisine.
fix:
add new unknown category


In [ ]:
# check if any actual_delivery_time was recorded before created_at
(df["actual_delivery_time"] < df["created_at"]).sum()
# good. all actual_delivery_time happened after created_at

np.int64(0)

## **Target Distribution**

In [2]:
df["created_at"]=pd.to_datetime(df["created_at"])
df["actual_delivery_time"]=pd.to_datetime(df["actual_delivery_time"])

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 16 columns):
 #   Column                                        Non-Null Count   Dtype         
---  ------                                        --------------   -----         
 0   market_id                                     196441 non-null  float64       
 1   created_at                                    197428 non-null  datetime64[us]
 2   actual_delivery_time                          197421 non-null  datetime64[us]
 3   store_id                                      197428 non-null  int64         
 4   store_primary_category                        192668 non-null  str           
 5   order_protocol                                196433 non-null  float64       
 6   total_items                                   197428 non-null  int64         
 7   subtotal                                      197428 non-null  int64         
 8   num_distinct_items                            197428 non-null  in

In [4]:
df["delivery_duration"] = (
    df["actual_delivery_time"] - df["created_at"]
).dt.total_seconds()
df.head()

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration,delivery_duration
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0,3779.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0,4024.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0,1781.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0,3075.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0,2390.0


In [17]:
df['delivery_duration'].describe(
    percentiles=[.01, .05, .10, .25, .50, .75, .90, .99, .999]
)

count    1.974210e+05
mean     2.908257e+03
std      1.922961e+04
min      1.010000e+02
1%       1.152000e+03
5%       1.492000e+03
10%      1.699000e+03
25%      2.104000e+03
50%      2.660000e+03
75%      3.381000e+03
90%      4.235000e+03
99%      6.474800e+03
99.9%    9.934860e+03
max      8.516859e+06
Name: delivery_duration, dtype: float64

You have:
- 194,721 records with non-null delivery_duration.
- 50th percentile (median): 2,660 seconds ≈ 44.3 minutes.
- 90th percentile: 4,235 seconds ≈ 70.6 minutes.
- 99th percentile: 6,474.8 seconds ≈ 107.9 minutes.
- 99.9th percentile: 9,934.86 seconds ≈ 165.6 minutes.
- Maximum: 8,516,859 seconds ≈ 98.6 days.

90% of deliverie took at most 4,235 seconds.

The distribution exhibits a long right tail, with the maximum duration reaching 8,516,859 seconds (approximately 98.6 days).

The extreme maximum warrants investigation. The corresponding record has an order creation date in October 2014 and a delivery date in January 2015.

However, the dataset documentation states that noise was added to obfuscate certain business details. Consequently, the original operational validity of extreme observations cannot be independently established.

These observations will be flagged for further data-quality assessment rather than automatically classified as erroneous.



## Invalid values check

In [ ]:
# zero or negative duraion
(df['delivery_duration'] <= 0).sum()

np.int64(0)

In [19]:
# categorical value consistency. only 1 str feature
df["store_primary_category"].nunique()

74

In [20]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
market_id,196441.0,2.978706,1.0,2.0,3.0,4.0,6.0,1.524867
created_at,197428,2015-02-04 22:00:09.537963,2014-10-19 05:24:15,2015-01-29 02:32:42,2015-02-05 03:29:09.500000,2015-02-12 01:39:18.500000,2015-02-18 06:00:44,NaN
actual_delivery_time,197421,2015-02-04 22:48:23.348914,2015-01-21 15:58:11,2015-01-29 03:22:29,2015-02-05 04:40:41,2015-02-12 02:25:26,2015-02-19 22:45:31,NaN
store_id,197428.0,3530.510272,1.0,1686.0,3592.0,5299.0,6987.0,2053.496711
order_protocol,196433.0,2.882352,1.0,1.0,3.0,4.0,7.0,1.503771
total_items,197428.0,3.196391,1.0,2.0,3.0,4.0,411.0,2.666546
subtotal,197428.0,2682.331402,0.0,1400.0,2200.0,3395.0,27100.0,1823.093688
num_distinct_items,197428.0,2.670791,1.0,1.0,2.0,3.0,20.0,1.630255
min_item_price,197428.0,686.21847,-86.0,299.0,595.0,949.0,14700.0,522.038648
max_item_price,197428.0,1159.58863,0.0,800.0,1095.0,1395.0,14700.0,558.411377


In [ ]:
invalid_features = [
    "min_item_price",
    "total_onshift_dashers",
    "total_busy_dashers",
    "total_outstanding_orders",
]


Cleaning steps:
- convert date features to pd.datetime dtype

- missing `estimated_store_to_consumer_driving_duration`
fix:
1. Create a missingness indicator.
2. Inpute fallback hierarchy: store-level median -> maket-level mdeian -> global median
3. Calculate all medians using training data only.
enough obersavation = 5 (not saying it's optimal thorugh experimentation i would get optimal.)

- missing `market_id`
fix:
add new unknown category

- missing `actual_delivery_time`
fix:
drop those 7 rows.

- missing `order_protocol`
fix:
add new unknown category

- missing `store_primary_category`
fix:
add new unknown category

- For `nonegative_features`
For features whose documented semantics require non-negative values, negative observations are treated as invalid and converted to missing values. Missing or invalid values are imputed using the median of valid training observations.
nonnegative_features = [
    "min_item_price",
    "total_onshift_dashers",
    "total_busy_dashers",
    "total_outstanding_orders",
]
1. Replace negative values with missing values.
2. Impute using a median calculated from valid training observations(without -ve values).
3. Apply the fitted imputation to validation and test data.

- `store_primary_category` has 74 unique categories
fix:
Target encoding

order_protocol, market_id, store_id - are actually objects but represented as numbers in the data
order_protocol - array([ 1.,  2.,  3.,  4., nan,  5.,  6.,  7.]) - can do onehot encode
market_id - array([ 1.,  2.,  3.,  4., nan,  5.,  6.]) - can do onehot encode

store_id - 6743 unique stores 
- drop store_id, store_primary_category from modeling for now. check whether adding it improves model performance. how to add it: represent each store_id with it's historical average delivery duration.

In [7]:
df.loc[
    df["min_item_price"] < 0,
    [
        "created_at",
        "subtotal",
        "num_distinct_items",
        "min_item_price",
        "max_item_price",
    ],
]
# 2552 for <= 0
# 13 for < 0

,created_at,subtotal,num_distinct_items,min_item_price,max_item_price
790,2015-01-29 02:59:28,1003,3,-86,1066
1351,2015-02-06 05:26:55,1786,4,-1,488
46369,2015-02-08 23:20:59,1636,5,-35,514
46378,2015-02-08 06:02:12,1822,4,-48,433
77238,2015-02-14 22:01:48,3099,1,-31,52
91319,2015-02-15 20:46:57,1601,5,-13,194
104173,2015-01-25 19:18:20,6095,7,-30,1211
119785,2015-01-24 03:45:37,1938,8,-9,594
119854,2015-02-10 03:31:22,2585,3,-7,1235
165821,2015-02-02 20:24:15,1646,3,-6,297


Modeling data pipeline implementation blueprint.
Pipeline A — Data cleaning and target preparation
1. Data ingestion
   - Load raw CSV.
   - Validate required columns.
   - Validate column names and schema.
   - Preserve the original data.
2. Timestamp parsing and validation
   - Parse created_at and actual_delivery_time.
   - Check parsing failures.
   - Check missing timestamps.
   - Check temporal consistency.
3. Data cleaning and target construction
   - Convert invalid negative counts and prices to missing.
   - Calculate delivery_duration in seconds.
        - Check whether delivery timestamps precede creation timestamps.
   - Exclude observations with missing targets from supervised modeling.
        - Missing target exclusion  

        Pass Clean, labeled dataset

Pipeline B — Feature engineering
4. Train-validation-test split
   - Split chronologically using created_at.
   - Keep the test set untouched.
   - Preserve y and the identifiers needed for later analysis.
5. Deterministic feature engineering
   - Extract calendar features from created_at.
   - Consider local Pacific time for hour and weekday.
   - Create missingness indicators.
   - Select baseline features.
   - Exclude target and unavailable-at-prediction-time columns from X.

   Pas Train / validation / test split

Pipeline C — Fitted preprocessing
6. Train-fitted preprocessing
   - Fit imputation statistics on training data.
   - Fit categorical encoders on training data.
   - Handle unseen categories.
   - Transform validation and test data using the fitted objects.
7. Model-ready validation
   - Check feature names and order.
   - Check dtypes.
   - Check missing-value expectations.
   - Check that train, validation, and test feature schemas match.
   - Produce X_train, y_train, X_val, y_val.
   - Keep test data separate for final evaluation.


> can call Pipeline B and C together your feature preprocessing pipeline. But keeping deterministic feature engineering separate from fitted preprocessing makes the implementation easier to reason about.
> keep raw data, cleaned data, engineered features, and fitted preprocessing distinct


Then perform temporal cross-validation inside the 80% development data.

This gives you:
- Training data for fitting models.
- Validation folds for model selection and tuning.
- A final test set for estimating performance on future orders.